In [ ]:
import os
import fastf1
import pandas as pd
import numpy as np

# 1. 환경 설정 및 캐시 활성화
cache_dir = 'f1_cache'
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)
fastf1.Cache.enable_cache(cache_dir)

circuit_name = 'Belgium'

# ----------------------------------------------------
# [TRACK 1] 서킷의 역사적 매크로 데이터 수집 (과거 20개 시즌 타겟)
# ----------------------------------------------------
# 2006년부터 2025년까지의 벨기에 GP 역사적 변수 추출 (2006년은 스파 미개최로 예외처리 발생 가능)
macro_years = list(range(2006, 2026)) 
sc_events = []
sc_trigger_laps = []

print("📚 [TRACK 1] 스파 서킷 20개년 매크로 통계 수집 시작...")
for year in macro_years:
    try:
        session = fastf1.get_session(year, circuit_name, 'R')
        session.load(laps=True, telemetry=False, weather=False) # 가볍게 로드
        laps = session.laps
        
        # 세이프티 카(4) 또는 버추얼 세이프티 카(6) 발동 랩 수 카운트
        sc_laps = laps[laps['TrackStatus'].str.contains('4|6', na=False)]['LapNumber'].unique()
        total_laps = laps['LapNumber'].max()

        # 2. 경기당 SC 발생 여부 (한 번이라도 터졌으면 1, 아니면 0)
        sc_lap_numbers = len(sc_laps)
        has_sc = 1 if sc_lap_numbers > 0 else 0
        
        if total_laps > 0:
            sc_events.append({
                'Year': year, 
                'TotalLaps': total_laps, 
                'SCLaps': sc_lap_numbers,
                'SC_Occurred': has_sc # 경기당 발생 여부 저장
            })

        if has_sc == 1:
            first_sc_lap = min(sc_laps)
            sc_trigger_laps.append(first_sc_lap)
        
    except Exception as e:
        # F1 스파 서킷이 개최되지 않은 해(예: 2006년)나 데이터 오류 자동 스킵
        continue

df_macro_sc = pd.DataFrame(sc_events)
historical_sc_prob = df_macro_sc['SCLaps'].sum() / df_macro_sc['TotalLaps'].sum() #랩 당 SC 발생 확률

race_sc_prob = df_macro_sc['SC_Occurred'].sum() / len(df_macro_sc) #경기당 SC 발생 확률

avg_sc_lap = np.mean(sc_trigger_laps) #SC 발동 시점 평균 랩
std_sc_lap = np.std(sc_trigger_laps) #SC 발동 시점 랩의 표준편차

# ----------------------------------------------------
# [TRACK 2] 최근 4개년 19인 타이어/페이스 마이크로 데이터 수집 (가중치 적용)
# ----------------------------------------------------
micro_years = [2022, 2023, 2024, 2025]
all_laps_list = []
pit_losses = []

print("\n🏎️ [TRACK 2] 최근 4개년 전수조사 및 피트 손실 계산 시작...")
weight_dict = {2025: 1.0, 2024: 0.7, 2023: 0.4, 2022: 0.1}

for year in micro_years:
    try:
        session = fastf1.get_session(year, circuit_name, 'R')
        session.load() # 타이어 마모 분석용 상세 로드
        laps = session.laps
        
        # 1. 정상 주행 데이터 정제 (가중치 포함)
        cond_normal = laps['TrackStatus'] == '1'
        cond_no_pit = (laps['PitInTime'].isna()) & (laps['PitOutTime'].isna())
        cleaned_laps = laps[cond_normal & cond_no_pit].copy()
        
        cleaned_laps['Weight'] = weight_dict[year]
        cleaned_laps['Year'] = year
        cleaned_laps['LapTimeSeconds'] = cleaned_laps['LapTime'].dt.total_seconds()
        
        features = ['Year', 'Weight', 'Driver', 'Team', 'LapNumber', 'LapTimeSeconds', 'Compound', 'TyreLife', 'Stint']
        all_laps_list.append(cleaned_laps[features].dropna())
        
        # 2. 피트레인 감속 손실 시간 상세 계산 (정밀도 증가)
        pit_laps = laps[laps['PitInTime'].notna() | laps['PitOutTime'].notna()]
        for idx, row in pit_laps.iterrows():
            driver = row['Driver']
            avg_normal = cleaned_laps[cleaned_laps['Driver'] == driver]['LapTimeSeconds'].mean()
            pit_lap_time = row['LapTime'].total_seconds() if pd.notna(row['LapTime']) else None
            
            if pit_lap_time and avg_normal:
                loss = pit_lap_time - avg_normal
                if 15 < loss < 40: # 아웃라이어 제거 플러그
                    pit_losses.append(loss)
    except Exception as e:
        print(f"⚠️ {year}년 마이크로 데이터 수집 실패: {e}")

df_laps = pd.concat(all_laps_list, ignore_index=True)
mean_pit_loss = np.mean(pit_losses)

# ----------------------------------------------------
# 최종 리포트 출력
# ----------------------------------------------------
print("\n🎯 [Step 1 완료] 완벽한 전략 데이터베이스 구축!")
print(f"1. 20개년 스파 역사상 '전체 레이스 중 SC주행' 리스크 확률: {historical_sc_prob * 100:.2f}%")
print(f"2. 실전 피트 손실 통계 (Pit Lane Loss): {mean_pit_loss:.2f} 초")
print(f"3. 머신러닝용 최근 4개년 데이터 로우 수: {len(df_laps)}개")
print(f"4. 경기당 세이프티 카(SC/VSC) 발생 확률: {race_sc_prob * 100:.2f}%")
print(f"5. SC 발동 시 평균 타이밍: 레이스 시작 후 {avg_sc_lap:.1f}바퀴 시점 (±{std_sc_lap:.1f}랩)")


events      WARNING 	Correcting user input 'Belgium' to 'Australian Grand Prix'


📚 [TRACK 1] 스파 서킷 20개년 매크로 통계 수집 시작...


core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
logger      WARNING 	Failed to load session info data!
core        WARNING 	Cannot load laps, telemetry, weather, and message data because the relevant API is not supported for this session.
core           INFO 	Finished loading data for 22 drivers: ['1', '3', '7', '16', '2', '17', '11', '14', '21', '12', '19', '22', '23', '4', '18', '20', '5', '9', '15', '8', '10', '6']
core           INFO 	Loading data for Belgian Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
logger      WARNING 	Failed to load session info data!
core        WARNING 	Cannot load laps, telemetry, weather, and message data because the relevant API is not supported for this session.
core           INFO 	Finished lo


🏎️ [TRACK 2] 최근 4개년 전수조사 및 피트 손실 계산 시작...


core        WARNING 	Fixed incorrect tyre stint information for driver '22'
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '63', '14', '16', '31', '5', '10', '23', '18', '4', '22', '24', '3', '20', '47', '6', '77', '44']
core           INFO 	Loading data for Belgian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing ti


🎯 [Ultimate Step 1 완료] 완벽한 전략 데이터베이스 구축!
1. 20개년 스파 역사상 '전체 레이스 중 SC주행' 리스크 확률: 6.43%
2. 실전 피트 손실 통계 (Pit Lane Loss): 18.88 초
3. 머신러닝용 최근 4개년 데이터 로우 수: 2744개
경기당 세이프티 카(SC/VSC) 발생 확률: 62.50%
SC 발동 시 평균 타이밍: 레이스 시작 후 3.0바퀴 시점 (±3.5랩)
